In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import pandas_ta as ta
import math
# from tqdm import tqdm
# import gc
# import time
import json
from pprint import pprint
# import pandas_ta
# import talib
import pickle
# from position_tools import calculate_trades, calculate_positions, count_since_last_signal

# from pklibs import *

import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

import talib

# from pklib.strategy import *
from pklib.utilities import *
# from pklib.pkindicators import calculate_zigzag
from pklib.indicators import *

# from sklearn.metrics import confusion_matrix
# from sklearn.metrics import confusion_matrix
# from sklearn.metrics import precision_score, recall_score

# import seaborn as sns
# from pklib.rl import *

### IMPORTANT
# pip install numpy==1.26.4 pandas==2.2.1 --force-reinstall

In [11]:
import dotenv
import os

# Reload the variables in your '.env' file (override the existing variables)
dotenv.load_dotenv(".env", override=True)

# 'MY_VAR' is refreshed now
print('HIP_VISIBLE_DEVICES = ', os.environ.get('HIP_VISIBLE_DEVICES')) # MY_VAR = HELLO_BOB

HIP_VISIBLE_DEVICES =  0


In [12]:

import torch as T
import torch.nn as nn
import torch.optim as optim

In [29]:

# from sklearn.cross_validation import train_test_split
# from sklearn.metrics import confusion_matrix
# from sklearn.metrics import precision_score, recall_score

def confusion_matrix_df(y_true, y_pred):
    # Create a 2x2 matrix for binary classification
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    # Create the confusion matrix as a DataFrame
    cm = np.array([[tn, fp], [fn, tp]])
    cm_df = pd.DataFrame(cm, 
                         index=["True Negative", "True Positive"],  # Rows (True Labels)
                         columns=["Predicted Negative", "Predicted Positive"])  # Columns (Predicted Labels)
    
    return cm_df

def precision_recall(cm):
    tn, fp, fn, tp = cm.ravel()  # For binary classification

    # Precision: TP / (TP + FP)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    # Recall: TP / (TP + FN)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    return precision, recall


def plot_confusion_matrix(cm, labels):
    fig, ax = plt.subplots()
    cax = ax.matshow(cm, cmap=plt.cm.Blues)
    fig.colorbar(cax)

    # Add labels to the axes
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    # Rotate the x-axis labels to prevent overlap
    plt.xticks(rotation=45)

    # Annotate the confusion matrix with the numbers
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")

    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

# Example usage
# labels = ['Class 0', 'Class 1'] 

def train_test_split(X, y, test_size=0.25, random_state=None, shuffle=True):
    # Ensure input is a NumPy array
    X = np.array(X)
    y = np.array(y)
    
    # Set the random seed for reproducibility
    if random_state is not None:
        np.random.seed(random_state)
    
    # Shuffle the data
    if shuffle:
        indices = np.random.permutation(len(X))
        X = X[indices]
        y = y[indices]
    
    # Compute the split index
    split_idx = int(len(X) * (1 - test_size))
    
    # Split the data into training and testing sets
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    return X_train, X_test, y_train, y_test

In [74]:

lookback_window_size = 14
trading_mode = 'long_only'# Wrap the environment

# params = {'conversion_periods': 110, 'base_periods': 301, 'lagging_span2_periods': 250, 'displacement': 180}
# params = {'conversion_periods': 20, 'base_periods': 60, 'lagging_span2_periods': 120, 'displacement': 30}
# Parameters
exchange = 'binance'; asset = 'BTC'; quote = 'USDT'; nhours = 8
train_size= 0.7



df = load_candles(exchange, asset, quote, '4h').ffill().bfill()
# df = df.resample(f'{nhours}H').agg({'open': 'first','high': 'max','low': 'min','close': 'last','volume': 'sum'}).ffill().bfill()
df.drop('volume', axis=1, inplace=True)
# add_ichimoku_cloud_indicator(df, params={'conversion_periods': 20, 'base_periods': 60, 'lagging_span2_periods': 120, 'displacement': 30})
# add_ichimoku_cloud_indicator(df, params={'conversion_periods': 110, 'base_periods': 301, 'lagging_span2_periods': 250, 'displacement': 180})
add_bollinger_bands(df, periods=[3,5,7,9,14,21,50,100], multiplier=2)
# add_rsi_columns(df,periods=[9,14,21])
# add_adx_columns(df,periods=[9,14,21])
# add_mom_columns(df,periods=[3,5,7,9,14,21,50,100])
# add_ema_columns(df,periods=[3,5,7,9,7,14,21,50,100])
add_sma_columns(df,periods=[3,5,7,9,7,14,21,50,100])
# add_std_columns(df,periods=[3,5,7,9,7,14,21])
# log_price_over_ma_columns(df,periods=[3,5,9,14,21,50])
# add_sma_columns(df,periods=[14,21,50,100,200])
# add_std_columns(df,periods=[14,21,50,100,200])
# add_donchian_columns(df,periods=[9,14,21,50])
# add_rolling_max_columns(df,periods=[3,5,7,9,7,14,21,50,100], column="close")
# add_rolling_min_columns(df,periods=[3,5,7,9,7,14,21,50,100], column="close")
# add_rolling_max_columns(df,periods=[9,14,21,50], column="high")
# add_rolling_min_columns(df,periods=[9,14,21,50], column="low")

# add_shifted_columns(df, periods=[14,21,50,100,200,400], columns=['open','high','low','close'], suffix="_SH")

# df = df.dropna()
# df = df['2021':]#.iloc[:10000]
# print(f'DF shape: {df.shape}')
# df_logprice = df.close.apply(np.log).ffill().bfill()

# (train_df_features, train_df_log_prices), (test_df_features, test_df_log_prices) = split_dataframes([df, df_logprice], train_size=train_size)

# add_shifted_columns(df, periods=[1,2,3], columns=['open','high','low','close'], suffix="_SH")
df = df.dropna()
dfs = df.apply(np.log)
# X = df.subtract(df.close, axis=0)
X = dfs.subtract(dfs.close, axis=0)
lookahead = 5
# Y = (df.low.shift(-1) > df.low).fillna(False).astype(int)
fwd_min = dfs.low.rolling(lookahead).min().shift(-lookahead)
fwd_max = dfs.high.rolling(lookahead).max().shift(-lookahead)
fwd_rwd = (fwd_max - dfs.close)
fwd_rsk = (dfs.close - fwd_min)
fwd_rwd_pct = np.expm1(fwd_rwd)
fwd_rsk_pct = np.expm1(fwd_rsk)

Y = (fwd_min > dfs.low.rolling(2).min()).fillna(False)
Y = Y & (fwd_rwd_pct >= 2*(- fwd_rsk_pct))
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.5, random_state=42, shuffle=False)
X
# n_train,n_features = train_df_features.shape


,open,high,low,close,BB_3_SMA,BB_3_STD,BB_3_Upper,BB_3_Lower,BB_3_DchUp,BB_3_DchDn,...,BB_100_DchUp,BB_100_DchDn,SMA_3,SMA_5,SMA_7,SMA_9,SMA_14,SMA_21,SMA_50,SMA_100
timestamp,,,,,,,,,,,,,,,,,,,,,
2017-09-02 16:00:00,-0.009950,0.009975,-0.029784,0.0,0.005209,-3.944357,0.043012,-0.034079,0.026313,-0.011055,...,0.069616,-0.171750,0.005209,0.023531,0.036255,0.041617,0.041429,0.032023,-0.005510,-0.044965
2017-09-02 20:00:00,0.008379,0.008421,-0.042310,0.0,0.001912,-5.152357,0.013395,-0.009704,0.008379,-0.002676,...,0.077995,-0.163371,0.001912,0.018131,0.034159,0.042813,0.047120,0.040134,0.003216,-0.036303
2017-09-03 00:00:00,-0.013448,0.010636,-0.022499,0.0,-0.011531,-4.532950,0.009983,-0.033518,0.000000,-0.021546,...,0.056450,-0.184917,-0.011531,-0.009059,0.004387,0.015675,0.023604,0.018822,-0.017161,-0.057519
2017-09-03 04:00:00,-0.007841,0.020115,-0.007841,0.0,-0.014537,-4.113612,0.018101,-0.048276,0.000000,-0.032716,...,0.045280,-0.196087,-0.014537,-0.020634,-0.012001,-0.000111,0.010844,0.008510,-0.026818,-0.068067
2017-09-03 08:00:00,0.027379,0.041238,-0.006530,0.0,0.016099,-4.188136,0.045526,-0.014221,0.029622,0.000000,...,0.074901,-0.166466,0.016099,0.010128,0.010975,0.021084,0.036962,0.037425,0.003640,-0.038073
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-23 00:00:00,-0.003596,0.006169,-0.004154,0.0,-0.002770,-6.007651,0.002152,-0.007715,0.000000,-0.004720,...,0.017872,-0.101239,-0.002770,0.000179,0.000944,0.003551,-0.004854,-0.007458,-0.020028,-0.021060
2024-08-23 04:00:00,-0.007824,0.004875,-0.008000,0.0,-0.006403,-5.148622,0.005218,-0.018162,0.000000,-0.011420,...,0.010048,-0.109063,-0.006403,-0.008527,-0.005655,-0.005188,-0.010246,-0.013620,-0.027049,-0.028098
2024-08-23 08:00:00,0.005379,0.009762,-0.000833,0.0,0.000983,-5.519109,0.008962,-0.007060,0.005379,-0.002446,...,0.015427,-0.103684,0.000983,-0.002045,-0.000847,-0.000568,-0.003189,-0.007212,-0.020897,-0.022050


In [75]:
y_train.sum()

tensor(13084., device='cuda:0')

In [76]:
# Y
import numpy as np
import torch
window_size = 4

def create_rolling_windows(data, window_size=window_size):
    """
    Create rolling windows of a given size from the input data.
    
    Args:
    - data (np.array): The input data of shape (sequence_length, features)
    - window_size (int): Number of past samples to include at every timestep
    
    Returns:
    - np.array: The data transformed into rolling windows of shape (new_sequence_length, window_size, features)
    """
    rolling_data = []
    
    # Create the windows
    for i in range(len(data) - window_size + 1):
        window = data[i:i + window_size]  # Take a window of 4 samples
        rolling_data.append(window)
    
    return np.array(rolling_data)


device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        
# Example: X_train is the input dataframe
X_train_np = X_train  # Convert the DataFrame to a numpy array
X_train_windows = create_rolling_windows(X_train_np, window_size=window_size)

# Convert back to a PyTorch tensor
x_train = torch.tensor(X_train_windows, dtype=torch.float32).to(device)


# Example: X_train is the input dataframe
X_test_np = X_test  # Convert the DataFrame to a numpy array
X_test_windows = create_rolling_windows(X_test_np, window_size=window_size)
# Convert back to a PyTorch tensor
x_test = torch.tensor(X_test_windows, dtype=torch.float32).to(device)

# Example: y_train should have the shape (batch_size, sequence_length, 1), where sequence_length = 4 (window size)
y_train = torch.tensor(Y_train[3:], dtype=torch.float32).unsqueeze(1).repeat(1, 4).unsqueeze(-1).to(device)

# Similarly, reshape y_test
y_test = torch.tensor(Y_test[3:], dtype=torch.float32).unsqueeze(1).repeat(1, 4).unsqueeze(-1).to(device)


In [77]:
y_train.shape, pos_weight, Y_train.shape
# device

(torch.Size([7630, 4, 1]), tensor([1.3326], device='cuda:0'), (7633,))

In [82]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# Define the model class with proper weight initialization
class RollingWindowLSTMBinaryClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super(RollingWindowLSTMBinaryClassifier, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layer with dropout
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        
        # Fully connected layer to output a single value for binary classification
        self.fc = nn.Linear(hidden_size, 1)
    
        self.to(device)
        
        # Initialize weights
        self.init_weights()
        
    def init_weights(self):
        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                nn.init.zeros_(param.data)
                
        nn.init.xavier_uniform_(self.fc.weight.data)
        nn.init.zeros_(self.fc.bias.data)
    
    def forward(self, x):
        h_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        c_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        
        # Forward pass through LSTM
        out, (hn, cn) = self.lstm(x, (h_0, c_0))
        
        # Pass through fully connected layer
        out = self.fc(out)
        
        return out

# Define the evaluation function
def evaluate_model(model, x_data, y_data, dataset_name="Dataset"):
    model.eval()  # Set the model to evaluation mode
    
    with torch.no_grad():  # Disable gradient computation
        # Make predictions
        outputs = model(x_data.to(device))
        predicted = torch.sigmoid(outputs) > 0.5  # Convert logits to binary predictions
        
        # Flatten predictions and ground truth
        predicted_flat = predicted.view(-1).cpu().numpy()
        y_flat = y_data.view(-1).cpu().numpy()
        
        # Compute confusion matrix
        cm = confusion_matrix(y_flat, predicted_flat)
        
        # Compute precision and recall
        precision, recall, _, _ = precision_recall_fscore_support(y_flat, predicted_flat, average='binary')
        
        # Display the confusion matrix
        cm_df = pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1'])
        print(f"\nConfusion Matrix - {dataset_name}:\n{cm_df}")
        
        # Output precision and recall
        print(f"Precision ({dataset_name}): {precision:.4f}")
        print(f"Recall ({dataset_name}): {recall:.4f}")
        
        return precision, recall

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Parameters
input_size = x_train.size(2)  # Number of features in X_train
hidden_size = input_size * 4   # Number of hidden units in the LSTM
num_layers = 3                 # Number of LSTM layers

# Initialize the model
model = RollingWindowLSTMBinaryClassifier(input_size, hidden_size, num_layers, dropout=0.3)

# Calculate class weights
# Y
# y_train = y_train.view(-1)  # Ensure Y_train is 1D
num_neg = (y_train == 0).sum().item()
num_pos = (y_train == 1).sum().item()

if num_pos == 0:
    raise ValueError("No positive samples in Y_train, cannot compute pos_weight.")

pos_weight = torch.tensor([num_neg / num_pos]).to(device)

# Define the loss function with pos_weight
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer with a lower learning rate
optimizer = optim.Adam(model.parameters(), lr=1e-2)

# Optional: Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.80)

# Normalize input data (if not already normalized)
# Example using torch
mean = x_train.mean(dim=(0,1), keepdim=True)
std = x_train.std(dim=(0,1), keepdim=True)
x_train = (x_train - mean) / (std + 1e-8)  # Add epsilon to avoid division by zero

# Training loop
num_epochs = 500
for epoch in range(num_epochs):
    model.train()
    
    # Zero the gradients
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(x_train.to(device))
    
    # Check for NaNs or Infs in outputs
    if torch.isnan(outputs).any() or torch.isinf(outputs).any():
        print(f"Epoch {epoch + 1}: Model outputs contain NaNs or Infs. Stopping training.")
        break
    
    # Compute loss
    loss = criterion(outputs, y_train.to(device))
    
    # Check for NaNs in loss
    if torch.isnan(loss):
        print(f"Epoch {epoch + 1}: Loss is NaN. Stopping training.")
        break
    
    # Backward pass
    loss.backward()
    
    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    # Optimization step
    optimizer.step()
    
    # Scheduler step
    scheduler.step()
    
    if (epoch + 1) % 10 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}, Learning Rate: {current_lr:.6f}')

# Evaluate the model after training
evaluate_model(model, x_train, y_train, dataset_name="Train")


Epoch [10/500], Loss: 0.7931, Learning Rate: 0.010000
Epoch [20/500], Loss: 0.7772, Learning Rate: 0.010000
Epoch [30/500], Loss: 0.7583, Learning Rate: 0.010000
Epoch [40/500], Loss: 0.7317, Learning Rate: 0.010000
Epoch [50/500], Loss: 0.7042, Learning Rate: 0.010000
Epoch [60/500], Loss: 0.7070, Learning Rate: 0.010000
Epoch [70/500], Loss: 0.6623, Learning Rate: 0.010000
Epoch [80/500], Loss: 0.6394, Learning Rate: 0.010000
Epoch [90/500], Loss: 0.5717, Learning Rate: 0.010000
Epoch [100/500], Loss: 0.5214, Learning Rate: 0.008000
Epoch [110/500], Loss: 0.4525, Learning Rate: 0.008000
Epoch [120/500], Loss: 0.4654, Learning Rate: 0.008000
Epoch [130/500], Loss: 0.4047, Learning Rate: 0.008000
Epoch [140/500], Loss: 0.3356, Learning Rate: 0.008000
Epoch [150/500], Loss: 0.3516, Learning Rate: 0.008000
Epoch [160/500], Loss: 0.2949, Learning Rate: 0.008000
Epoch [170/500], Loss: 0.2987, Learning Rate: 0.008000
Epoch [180/500], Loss: 0.2414, Learning Rate: 0.008000
Epoch [190/500], Lo

(0.9983197128236462, 0.999006420055029)

In [83]:

# Evaluate on train data
train_precision, train_recall = evaluate_model(model, x_train, y_train, dataset_name="Train")

# Evaluate on test data
test_precision, test_recall = evaluate_model(model, x_test, y_test, dataset_name="Test")


Confusion Matrix - Train:
          Predicted 0  Predicted 1
Actual 0        17414           22
Actual 1           13        13071
Precision (Train): 0.9983
Recall (Train): 0.9990

Confusion Matrix - Test:
          Predicted 0  Predicted 1
Actual 0         7449        11115
Actual 1         4619         7341
Precision (Test): 0.3978
Recall (Test): 0.6138


In [24]:
x_train.shape, X_train.shape, device


(torch.Size([7680, 4, 4]), (7683, 4), device(type='cuda', index=0))

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim

# X_train is a DataFrame with rows as timestamps and columns as features
x = torch.tensor(X_train, dtype=torch.float32)  # Single sequence with multiple features
y = torch.tensor(Y_train, dtype=torch.float32)  # Convert Y_train to float32 for BCELoss

# Add batch dimension (which will be 1 for a single sequence) to fit LSTM input (batch_size, sequence_length, input_size)
x = x.unsqueeze(0)  # Shape becomes (1, sequence_length, input_size)
y = y.unsqueeze(0).unsqueeze(-1)  # Shape becomes (1, sequence_length, 1) for per-timestamp prediction

# Define the LSTM model for binary classification per timestamp
class LSTMBinaryClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMBinaryClassifier, self).__init__()
        
        # LSTM layer
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        
        # Fully connected layer to output a single value for each timestamp
        self.fc = nn.Linear(hidden_size, 1)
        
        # Sigmoid activation to output a probability
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Initialize hidden and cell states
        h_0 = torch.zeros(num_layers, x.size(0), hidden_size).to(x.device)  # Hidden state
        c_0 = torch.zeros(num_layers, x.size(0), hidden_size).to(x.device)  # Cell state
        
        # Forward pass through LSTM
        out, (hn, cn) = self.lstm(x, (h_0, c_0))
        
        # Pass through fully connected layer for each time step
        out = self.fc(out)  # Shape: (batch_size, sequence_length, 1)
        
        # Apply sigmoid to get probabilities
        out = self.sigmoid(out)
        
        return out

# Parameters
input_size = x.size(2)  # Number of features (columns in X_train)
hidden_size = 16        # Number of hidden units in the LSTM
num_layers = 3          # Number of LSTM layers

# Initialize the model
model = LSTMBinaryClassifier(input_size, hidden_size, num_layers)

# Loss and optimizer
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    
    # Zero the gradients
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(x)
    
    # Compute loss
    loss = criterion(outputs, y)
    
    # Backward pass and optimization
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

# Making Predictions after training
model.eval()
with torch.no_grad():
    outputs = model(x)
    predicted = (outputs > 0.5).float()  # Convert probabilities to binary (0 or 1)
    
    # Flatten predictions and ground truth to 1D arrays for confusion matrix
    predicted_flat = predicted.view(-1).cpu().numpy()  # Shape: (sequence_length,)
    y_flat = y.view(-1).cpu().numpy()  # Shape: (sequence_length,)
    
    # Compute confusion matrix
    cm = confusion_matrix(y_flat, predicted_flat)
    cm_df = pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1'])

    # Plot the confusion matrix using Seaborn
    plt.figure(figsize=(5,4))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix");plt.ylabel('Actual');plt.xlabel('Predicted');plt.show();
    # print("Confusion Matrix:\n", cm)
    # Compute precision and recall
    precision = precision_score(y_flat, predicted_flat)
    recall = recall_score(y_flat, predicted_flat)

    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")


Epoch [10/50], Loss: 0.5075
Epoch [20/50], Loss: 0.4445
Epoch [30/50], Loss: 0.3361
Epoch [40/50], Loss: 0.1844
Epoch [50/50], Loss: 0.0973


NameError: name 'confusion_matrix' is not defined

In [ ]:

# Create a DataFrame for a better display
cm_df = pd.DataFrame(cm_test, index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1'])

# Plot the confusion matrix using Seaborn
plt.figure(figsize=(5,4))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()
precision = precision_score(y_flat, predicted_flat)
recall = recall_score(y_flat, predicted_flat)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score
import torch

# Convert X_test and Y_test to tensors
x_test = torch.tensor(X_test.values, dtype=torch.float32).unsqueeze(0)  # Shape: (1, sequence_length, input_size)
y_test = torch.tensor(Y_test.values, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)  # Shape: (1, sequence_length, 1)

# Switch the model to evaluation mode
model.eval()

# Disable gradient computation for inference
with torch.no_grad():
    # Make predictions on X_test
    outputs_test = model(x_test)
    
    # Convert probabilities to binary (0 or 1)
    predicted_test = (outputs_test > 0.5).float()

    # Flatten predictions and ground truth for confusion matrix, precision, and recall
    predicted_flat_test = predicted_test.view(-1).cpu().numpy()  # Shape: (sequence_length,)
    y_flat_test = y_test.view(-1).cpu().numpy()  # Shape: (sequence_length,)

    # Compute confusion matrix
    cm_test = confusion_matrix(y_flat_test, predicted_flat_test)

    # Compute precision and recall
    precision_test = precision_score(y_flat_test, predicted_flat_test)
    recall_test = recall_score(y_flat_test, predicted_flat_test)

# Output results
print("Confusion Matrix (Test Data):\n", cm_test)
print(f"Precision (Test Data): {precision_test:.4f}")
print(f"Recall (Test Data): {recall_test:.4f}")


In [ ]:
device

In [ ]:

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"

# Data
# max_features = 4
# x_data = [
#     [[0], [1], [1], [0], [0], [0]],
#     [[0], [0], [0], [2], [2], [0]],
#     [[0], [0], [0], [0], [3], [3]],
#     [[0], [2], [2], [0], [0], [0]],
#     [[0], [0], [3], [3], [0], [0]],
#     [[0], [0], [0], [0], [1], [1]]
# ]
x = torch.tensor(X_train, dtype=torch.float32)
y = torch.tensor(Y_train, dtype=torch.bool)

# Convert labels to one-hot encoding
# y2 = torch.nn.functional.one_hot(y, max_features).to(torch.float32)
# print(y2)

# Model using a sequence
class LSTMLayer(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(LSTMLayer, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)

    def forward(self, x):
        out, _ = self.lstm(x)
        return out

model = nn.Sequential(
    LSTMLayer(input_size=1, hidden_size=128),
    nn.Dropout(p=0.2),
    nn.Flatten(),
    nn.Linear(128*6, 4),
    nn.Sigmoid()
)

# Check for GPU availability
model.to(device)
x, y2 = x.to(device), y2.to(device)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Train the model
print('Train...')
for epoch in range(200):
    optimizer.zero_grad()
    outputs = model(x)
    loss = criterion(outputs, y2)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/200], Loss: {loss.item():.4f}")


# Predictions
with torch.no_grad():
    outputs = model(x)
    predicted_classes = torch.argmax(outputs, 1)
    print(f"Predicted classes: {predicted_classes.cpu().numpy()}")
    print(f"Expected classes: {y.cpu().numpy()}")